<div style="padding:28px 32px;border:1px solid #d0d7de;border-radius:14px;">
  <div style="font-size:14px;letter-spacing:.08em;text-transform:uppercase;"><b>Complete Study Guide</b></div>
  <h1 style="margin:.35em 0 .15em 0;">Every Theory Behind the Backtesting Engine</h1>
  <p style="font-size:18px;margin:.2em 0 0 0;">
    Finance, statistics, research design, execution assumptions, and the exact Python lines that implement them.
  </p>
</div>

<br>

This notebook explains the **finished backtesting project**, not a simplified imitation of it.

It covers all four project strategies:

- Buy and hold
- Moving-average trend following
- Trailing-return momentum
- Rolling z-score mean reversion

It also explains every major engine assumption and metric:

- Daily adjusted market data
- Signals, target positions, and executed positions
- Long, cash, and short exposure
- Close-to-close returns
- Execution delay and look-ahead bias
- Turnover and transaction costs
- Compounding
- Trade extraction and reversals
- Total and annualized return
- Volatility
- Risk-free rates and excess returns
- Sharpe ratio
- Sortino ratio
- Drawdown
- Daily win rate
- Market exposure
- Trade win rate
- Profit factor
- Training versus testing data
- Indicator warm-up
- Parameter search
- Overfitting and data snooping
- Benchmarks
- CSV caching, exports, charts, and automated tests

> **Educational warning:** A backtest is a historical simulation. It is not evidence that future profits are guaranteed.

## How to use this notebook

Read it once from top to bottom, then use the final sections as revision material.

Code cells are divided into two types:

1. **Runnable demonstrations** using small synthetic datasets.
2. **Exact excerpts from the project**, followed by line-by-line explanations.

The notebook does not need internet access. Place it inside the finished project folder if you want the cells that import `backtester.py` and `strategies.py` to run.

## Contents

### Part I — The system you built
1. [The purpose of a backtest](#1-the-purpose-of-a-backtest)  
2. [Project architecture](#2-project-architecture)  
3. [The complete information flow](#3-the-complete-information-flow)  

### Part II — Market data and time series
4. [Ticker, index, ETF, and adjusted price](#4-ticker-index-etf-and-adjusted-price)  
5. [Daily bars and close-to-close modelling](#5-daily-bars-and-close-to-close-modelling)  
6. [Pandas time-series alignment and data cleaning](#6-pandas-time-series-alignment-and-data-cleaning)  
7. [Caching and reproducibility](#7-caching-and-reproducibility)  

### Part III — The mechanics of the backtester
8. [Prices and simple returns](#8-prices-and-simple-returns)  
9. [Signals, target positions, and executed positions](#9-signals-target-positions-and-executed-positions)  
10. [Long, cash, and short exposure](#10-long-cash-and-short-exposure)  
11. [Execution delay and look-ahead bias](#11-execution-delay-and-look-ahead-bias)  
12. [Gross strategy return](#12-gross-strategy-return)  
13. [Turnover and transaction costs](#13-turnover-and-transaction-costs)  
14. [Net returns and compounding](#14-net-returns-and-compounding)  
15. [Drawdown mechanics](#15-drawdown-mechanics)  
16. [Trade extraction](#16-trade-extraction)  

### Part IV — The strategies
17. [Buy and hold](#17-buy-and-hold)  
18. [Moving-average trend following](#18-moving-average-trend-following)  
19. [Momentum](#19-momentum)  
20. [Mean reversion and z-scores](#20-mean-reversion-and-z-scores)  

### Part V — Performance and risk statistics
21. [Final value and total return](#21-final-value-and-total-return)  
22. [Annualized return](#22-annualized-return)  
23. [Volatility](#23-volatility)  
24. [Risk-free rate and excess return](#24-risk-free-rate-and-excess-return)  
25. [Sharpe ratio](#25-sharpe-ratio)  
26. [Downside deviation and Sortino ratio](#26-downside-deviation-and-sortino-ratio)  
27. [Maximum drawdown](#27-maximum-drawdown)  
28. [Daily win rate](#28-daily-win-rate)  
29. [Market exposure](#29-market-exposure)  
30. [Orders, trades, and trade win rate](#30-orders-trades-and-trade-win-rate)  
31. [Average, best, and worst trade](#31-average-best-and-worst-trade)  
32. [Profit factor](#32-profit-factor)  

### Part VI — Research design
33. [Why a benchmark is necessary](#33-why-a-benchmark-is-necessary)  
34. [Training, testing, and out-of-sample evidence](#34-training-testing-and-out-of-sample-evidence)  
35. [Warm-up history and the test boundary](#35-warm-up-history-and-the-test-boundary)  
36. [Parameter grids](#36-parameter-grids)  
37. [Selection by Sharpe ratio](#37-selection-by-sharpe-ratio)  
38. [Minimum trade filters](#38-minimum-trade-filters)  
39. [Overfitting and multiple testing](#39-overfitting-and-multiple-testing)  
40. [Full-period results versus independent evidence](#40-full-period-results-versus-independent-evidence)  

### Part VII — Engineering and limitations
41. [Exports and charts](#41-exports-and-charts)  
42. [Automated tests](#42-automated-tests)  
43. [Validation and defensive programming](#43-validation-and-defensive-programming)  
44. [Every important modelling limitation](#44-every-important-modelling-limitation)  
45. [How a professional engine would improve this](#45-how-a-professional-engine-would-improve-this)  

### Revision
46. [Code-to-theory map](#46-code-to-theory-map)  
47. [Formula sheet](#47-formula-sheet)  
48. [Glossary](#48-glossary)  
49. [Revision questions](#49-revision-questions)  
50. [Answers](#50-answers)

In [1]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

print("Notebook setup complete.")

Notebook setup complete.


# Part I — The system you built

---

# 1. The purpose of a backtest

A **backtest** asks:

> If a strategy had followed a precise set of rules using only information available at the time, what would have happened historically?

A valid backtest must connect five layers:

1. **Market data** — what prices existed?
2. **Decision rule** — what did the strategy want to hold?
3. **Execution assumption** — when could the decision become a real position?
4. **Accounting** — what returns and costs followed?
5. **Evaluation** — how profitable and risky was the result?

The engine is not a prediction machine. It is a historical accounting system attached to a rule.

<div style="padding:14px 18px;border-left:5px solid #d29922;background:#0000;">
<b>Core distinction</b><br>
A strategy creates decisions. A backtester calculates the financial consequences of those decisions.
</div>

# 2. Project architecture

The finished project separates responsibilities:

| File | Responsibility |
|---|---|
| `market_data.py` | Downloads and caches adjusted prices |
| `strategies.py` | Converts prices into target-position signals |
| `backtester.py` | Applies execution timing, returns, costs, compounding, trades, and metrics |
| `research.py` | Runs parameter grids and groups strategy comparisons |
| `main.py` | Orchestrates the complete research workflow |
| `tests/` | Checks important engine behaviour with deterministic synthetic data |
| `outputs/` | Stores CSV results, JSON parameters, and PNG charts |

This separation is more than software style. It protects the research logic.

For example, the moving-average strategy should not secretly calculate its own portfolio return. It should only answer:

```text
What position do I want?
```

The engine independently answers:

```text
What return would that position have earned under the shared execution and cost rules?
```

# 3. The complete information flow

```text
yfinance
   ↓
Adjusted daily close prices
   ↓
Strategy indicator calculations
   ↓
Target position: -1, 0, or 1
   ↓
Execution delay
   ↓
Executed position
   ↓
Asset return × position
   ↓
Gross strategy return
   ↓
Turnover × transaction-cost rate
   ↓
Net strategy return
   ↓
Compounded portfolio value
   ↓
Drawdown and trade extraction
   ↓
Return, risk, activity, and trade metrics
   ↓
Training search and out-of-sample comparison
```

Every arrow contains an assumption. A strong research report states those assumptions explicitly.

# Part II — Market data and time series

---

# 4. Ticker, index, ETF, and adjusted price

## Ticker

A ticker is a symbol used to identify a security. The project defaults to:

```python
TICKER = "SPY"
```

`SPY` is an exchange-traded fund designed to track the S&P 500.

## Index versus ETF

The S&P 500 is an **index**: a calculated measure of a basket of companies.

`SPY` is an **ETF**: a fund that can be traded.

This matters because a backtest should ideally model something that could have been owned, not merely a theoretical index value.

## Adjusted price

The project requests:

```python
auto_adjust=True
```

The intention is to use prices adjusted for corporate actions such as stock splits and distributions. Without adjustment, a split can look like a catastrophic negative return even though shareholders did not lose that amount economically.

Adjusted data makes historical return calculations more meaningful, although the exact adjustment method still depends on the data provider.

### Exact market-data code

```python
data = yf.Ticker(ticker).history(
    start=start,
    end=end,
    interval="1d",
    auto_adjust=True,
    actions=False,
)

prices = data["Close"].dropna().astype(float)
```

Line by line:

- `interval="1d"` requests one observation per trading day.
- `auto_adjust=True` asks the provider for adjusted OHLC prices.
- `actions=False` does not request separate dividend and split tables.
- `data["Close"]` selects the adjusted daily closing series returned under these settings.
- `.dropna()` removes missing prices.
- `.astype(float)` ensures numerical calculations use floating-point values.

# 5. Daily bars and close-to-close modelling

The engine is a **daily-bar** engine. It does not model every trade or every second.

Its return is:

$$
r_t=\frac{P_t}{P_{t-1}}-1
$$

This is a **close-to-close return**: the movement from the previous daily close to the current daily close.

The position on date $t$ is multiplied by that close-to-close return.

This means the engine asks:

> What position was held over the interval from the previous close to the current close?

It does **not** model:

- the opening auction
- intraday highs and lows
- exact order time
- limit orders
- partial fills
- intraday stop-losses

<div style="padding:14px 18px;border-left:5px solid #1f6feb;background:#0000;">
<b>Timing interpretation</b><br>
With a one-period signal delay, a signal completed at the previous close becomes the position applied to the next close-to-close return. This is a simplified but internally consistent daily execution model.
</div>

# 6. Pandas time-series alignment and data cleaning

Financial calculations are only valid when observations refer to the same dates.

The engine cleans prices with:

```python
clean = pd.to_numeric(prices, errors="coerce").dropna().astype(float)
clean = clean[~clean.index.duplicated(keep="last")].sort_index()
```

Theory behind each line:

- `pd.to_numeric(..., errors="coerce")` converts values to numbers; invalid text becomes missing.
- `.dropna()` removes unusable observations.
- `.astype(float)` creates a consistent numerical type.
- `~clean.index.duplicated(keep="last")` removes duplicate dates, keeping the last record.
- `.sort_index()` ensures time runs in chronological order.

Chronological ordering is essential because `pct_change`, `shift`, `rolling`, and `cumprod` all assume row order represents time.

## Aligning signals to prices

The engine uses:

```python
positions = (
    pd.to_numeric(target_positions, errors="coerce")
    .reindex(price_index)
    .fillna(0.0)
    .astype(float)
)
```

`reindex(price_index)` forces the strategy decisions onto the exact price dates.

A missing decision becomes `0.0`, meaning cash.

This is a conservative default, but it is still an assumption: missing data is interpreted as no exposure rather than “carry the last signal forward.”

In [2]:
dates = pd.to_datetime(["2025-01-01", "2025-01-02", "2025-01-03"])
prices_example = pd.Series([100, 101, 103], index=dates, name="price")

signals_with_missing_date = pd.Series(
    [1, 0],
    index=pd.to_datetime(["2025-01-01", "2025-01-03"]),
    name="signal",
)

aligned = signals_with_missing_date.reindex(prices_example.index).fillna(0.0)

pd.DataFrame({
    "price": prices_example,
    "original_signal": signals_with_missing_date,
    "aligned_signal": aligned,
})

,price,original_signal,aligned_signal
2025-01-01,100,1.000000,1.000000
2025-01-02,101,NaN,0.000000
2025-01-03,103,0.000000,0.000000


# 7. Caching and reproducibility

The downloader saves prices to a CSV file:

```python
cache_path = cache_directory / f"{ticker}_{start}_{end}.csv"
prices.to_frame("Close").to_csv(cache_path)
```

On later runs:

```python
if cache_path.exists() and not refresh:
    cached = pd.read_csv(cache_path, index_col="Date", parse_dates=True)
```

Why cache data?

- Faster reruns
- Fewer repeated network requests
- More reproducible results during development
- A record of the exact downloaded series

But caching creates a risk:

> A cached file may become stale or may preserve an earlier provider revision.

That is why the project includes a `refresh` option.

## Time-zone normalization

The project removes timezone information:

```python
if isinstance(prices.index, pd.DatetimeIndex) and prices.index.tz is not None:
    prices.index = prices.index.tz_localize(None)
```

A timezone-aware and timezone-naive index may fail to compare cleanly.

The project is working with daily dates, so it normalizes the index to timezone-naive timestamps for simpler splitting and alignment.

# Part III — The mechanics of the backtester

---

# 8. Prices and simple returns

The engine calculates:

```python
asset_returns = prices.pct_change(fill_method=None).fillna(0.0)
```

`pct_change` computes:

$$
r_t=\frac{P_t}{P_{t-1}}-1
$$

The first observation has no previous price, so its return is missing. The engine replaces that first missing value with zero.

A return is stored as a decimal:

| Financial display | Python value |
|---:|---:|
| 1% | 0.01 |
| -4% | -0.04 |
| 12.5% | 0.125 |

In [3]:
prices_demo = pd.Series(
    [100.0, 102.0, 99.0, 103.0],
    index=pd.date_range("2025-01-01", periods=4),
    name="price",
)

asset_returns_demo = prices_demo.pct_change(fill_method=None).fillna(0.0)

pd.DataFrame({
    "price": prices_demo,
    "return_decimal": asset_returns_demo,
    "return_percent": asset_returns_demo * 100,
}).round(4)

,price,return_decimal,return_percent
2025-01-01,100.000000,0.000000,0.000000
2025-01-02,102.000000,0.020000,2.000000
2025-01-03,99.000000,-0.029400,-2.941200
2025-01-04,103.000000,0.040400,4.040400


## Why simple returns are used

Simple returns connect directly to wealth:

$$
V_t=V_{t-1}(1+r_t)
$$

Log returns are useful in statistical analysis, but simple returns are intuitive for portfolio accounting and can be compounded with `(1 + return).cumprod()`.

# 9. Signals, target positions, and executed positions

The strategy returns a **target position**.

Allowed values are:

| Target | Meaning |
|---:|---|
| 1 | Fully long |
| 0 | Cash |
| -1 | Fully short |

The target position is what the strategy wants.

The **executed position** is what the engine actually applies after the delay.

The project preserves both columns:

```python
"target_position": target_positions,
"position": positions,
```

Keeping both is important for auditing timing.

# 10. Long, cash, and short exposure

## Long: position = 1

A long position benefits from positive asset returns.

$$
r_{strategy}=1\times r_{asset}
$$

## Cash: position = 0

The portfolio does not receive the asset return.

$$
r_{strategy}=0\times r_{asset}=0
$$

Before transaction costs, cash earns zero in this engine.

## Short: position = -1

A short position benefits when the asset falls.

$$
r_{strategy}=-1\times r_{asset}
$$

If the asset return is $-5\%$, a fully short position earns approximately $+5\%$ before costs.

In [4]:
return_scenarios = pd.Series(
    [0.05, -0.05],
    index=["Asset rises 5%", "Asset falls 5%"],
)

pd.DataFrame({
    "asset_return": return_scenarios,
    "long_position": return_scenarios * 1,
    "cash_position": return_scenarios * 0,
    "short_position": return_scenarios * -1,
})

,asset_return,long_position,cash_position,short_position
Asset rises 5%,0.050000,0.050000,0.000000,-0.050000
Asset falls 5%,-0.050000,-0.050000,-0.000000,0.050000


## Important short-selling simplifications

The engine allows `-1`, but it does not model:

- stock-borrow availability
- borrow fees
- recalls
- short-sale regulations
- collateral and margin
- asymmetric losses
- forced liquidation

A short position can lose more than 100% economically if the underlying price rises enough. The engine only guards against a single daily net return of -100% or worse.

# 11. Execution delay and look-ahead bias

The key line is:

```python
positions = target_positions.shift(config.execution_delay).fillna(0.0)
```

With:

```python
execution_delay = 1
```

today's target becomes tomorrow's executed position.

Why?

A strategy using today's closing price cannot logically earn the return that ended at today's closing price.

Without a delay, the test could use the end of a period to decide the position for that same period. That is look-ahead bias.

In [5]:
signal_demo = pd.Series(
    [0, 1, 1, 0],
    index=pd.date_range("2025-01-01", periods=4),
    name="target_position",
)

executed_demo = signal_demo.shift(1).fillna(0.0)

pd.DataFrame({
    "target_position": signal_demo,
    "executed_position": executed_demo,
})

,target_position,executed_position
2025-01-01,0,0.000000
2025-01-02,1,0.000000
2025-01-03,1,1.000000
2025-01-04,0,1.000000


## What `execution_delay=0` means

A zero delay applies the target position to the same row's return.

That can be valid only if the signal was genuinely known before the modelled return interval began.

For strategies built from the same daily close used in `pct_change`, zero delay would usually be unrealistic.

# 12. Gross strategy return

The engine calculates:

```python
gross_strategy_returns = positions.mul(asset_returns)
```

Mathematically:

$$
r^{gross}_{s,t}=w_t r_{a,t}
$$

where:

- $w_t$ is the executed position
- $r_{a,t}$ is the asset return
- $r^{gross}_{s,t}$ is the strategy return before costs

This engine is **all-in**:

- `1` means 100% long
- `-1` means 100% short
- `0` means no asset exposure

It does not currently support 30%, 50%, or 150% positions.

In [6]:
asset_returns = pd.Series([0.02, -0.01, 0.03, -0.04])
positions = pd.Series([0.0, 1.0, 1.0, -1.0])

gross = positions * asset_returns

pd.DataFrame({
    "asset_return": asset_returns,
    "position": positions,
    "gross_strategy_return": gross,
})

,asset_return,position,gross_strategy_return
0,0.020000,0.000000,0.000000
1,-0.010000,1.000000,-0.010000
2,0.030000,1.000000,0.030000
3,-0.040000,-1.000000,0.040000


# 13. Turnover and transaction costs

The engine compares today's executed position with yesterday's:

```python
previous_positions = positions.shift(1).fillna(0.0)
turnover = positions.sub(previous_positions).abs()
```

Mathematically:

$$
\text{turnover}_t=|w_t-w_{t-1}|
$$

Then:

```python
transaction_costs = turnover.mul(config.transaction_cost_rate)
```

$$
\text{cost}_t=\text{turnover}_t \times c
$$

where $c$ is the proportional cost rate.

## Turnover examples

| Previous | Current | Turnover | Interpretation |
|---:|---:|---:|---|
| 0 | 1 | 1 | Enter long |
| 1 | 0 | 1 | Exit long |
| 0 | -1 | 1 | Enter short |
| -1 | 0 | 1 | Cover short |
| 1 | -1 | 2 | Sell long and establish short |
| -1 | 1 | 2 | Cover short and establish long |
| 1 | 1 | 0 | No trade |

A direct reversal has turnover `2` because two units of notional change occur.

In [7]:
positions_turnover = pd.Series(
    [0, 1, 1, 0, -1, 1],
    index=["Start", "Enter long", "Hold", "Cash", "Enter short", "Reverse long"],
    dtype=float,
)

previous = positions_turnover.shift(1).fillna(0.0)
turnover = (positions_turnover - previous).abs()

pd.DataFrame({
    "previous_position": previous,
    "position": positions_turnover,
    "turnover": turnover,
})

,previous_position,position,turnover
Start,0.000000,0.000000,0.000000
Enter long,0.000000,1.000000,1.000000
Hold,1.000000,1.000000,0.000000
Cash,1.000000,0.000000,1.000000
Enter short,0.000000,-1.000000,1.000000
Reverse long,-1.000000,1.000000,2.000000


## What the cost rate represents

The single rate is a simplified proxy for:

- commissions
- bid-ask spread
- slippage
- fees
- market impact

With `transaction_cost_rate = 0.001`, one unit of turnover costs 0.1% of portfolio value.

A direct reversal has turnover 2, so it costs 0.2% under this model.

<div style="padding:14px 18px;border-left:5px solid #cf222e;background:#0000;">
<b>Important limitation</b><br>
Real costs depend on asset liquidity, order size, volatility, time of day, broker, and execution method. A constant percentage is useful for education but not a complete market-impact model.
</div>

# 14. Net returns and compounding

Net strategy return:

```python
strategy_returns = gross_strategy_returns.sub(transaction_costs)
```

$$
r^{net}_{s,t}=r^{gross}_{s,t}-\text{cost}_t
$$

Portfolio value:

```python
portfolio = config.initial_capital * strategy_returns.add(1.0).cumprod()
```

$$
V_t=V_0\prod_{i=1}^{t}(1+r^{net}_{s,i})
$$

This is geometric compounding.

In [8]:
initial_capital = 10_000.0
net_returns = pd.Series([0.02, -0.01, 0.03, -0.002])

portfolio_values = initial_capital * (1 + net_returns).cumprod()

pd.DataFrame({
    "net_return": net_returns,
    "growth_multiplier": 1 + net_returns,
    "portfolio": portfolio_values,
})

,net_return,growth_multiplier,portfolio
0,0.020000,1.020000,"10,200.000000"
1,-0.010000,0.990000,"10,098.000000"
2,0.030000,1.030000,"10,400.940000"
3,-0.002000,0.998000,"10,380.138120"


## The insolvency guard

The engine checks:

```python
if (strategy_returns <= -1.0).any():
    raise ValueError(...)
```

A return of `-1.0` means a 100% loss.

Then:

$$
1+r=0
$$

The portfolio becomes zero. A return below -100% creates an economically nonsensical negative compounded wealth path for this simple model.

The guard forces the user to inspect extreme costs, prices, or leverage assumptions.

# 15. Drawdown mechanics

The project calculates:

```python
running_peak = portfolio.cummax()
drawdown = portfolio.div(running_peak).sub(1.0)
```

At each date:

$$
\text{drawdown}_t=\frac{V_t}{\max(V_1,\ldots,V_t)}-1
$$

Drawdown is zero whenever the portfolio is at a new high.

It is negative when the portfolio is below its previous peak.

In [9]:
portfolio_demo = pd.Series(
    [10_000, 12_000, 15_000, 13_500, 9_000, 11_000, 15_500],
    index=pd.date_range("2025-01-01", periods=7),
)

running_peak = portfolio_demo.cummax()
drawdown_demo = portfolio_demo / running_peak - 1

pd.DataFrame({
    "portfolio": portfolio_demo,
    "running_peak": running_peak,
    "drawdown": drawdown_demo,
    "drawdown_percent": drawdown_demo * 100,
})

,portfolio,running_peak,drawdown,drawdown_percent
2025-01-01,10000,10000,0.000000,0.000000
2025-01-02,12000,12000,0.000000,0.000000
2025-01-03,15000,15000,0.000000,0.000000
2025-01-04,13500,15000,-0.100000,-10.000000
2025-01-05,9000,15000,-0.400000,-40.000000
2025-01-06,11000,15000,-0.266667,-26.666667
2025-01-07,15500,15500,0.000000,0.000000


A maximum drawdown of -40% means the worst historical peak-to-trough loss was 40%.

A portfolio can be in cash and still remain in drawdown. Cash prevents additional asset exposure; it does not automatically recover a previous loss.

# 16. Trade extraction

The engine converts the daily position sequence into individual trades.

A trade stores:

- Entry date
- Exit date
- Long or short side
- Entry price
- Exit price
- Holding days
- Compounded net trade return
- Closed or open status

Trade return is calculated from the daily net returns:

```python
trade_return = np.prod(1.0 + accumulated_returns) - 1.0
```

This includes transaction costs because the engine accumulates `strategy_return`, not gross return.

## Entry, holding, exit, and reversal logic

### Entry

If the current side is cash and the position becomes non-zero, a trade opens.

### Hold

If the position remains on the same side, the day's return is appended.

### Exit

If the position becomes zero, the exit day's transaction cost is appended to the old trade before closing it.

### Direct reversal

If position changes directly from long to short or short to long:

- The old trade closes at the previous observation.
- The new trade begins today.
- The reversal day's cost is assigned to the new trade because today's return uses the new position.

This is a specific accounting convention. Other engines may allocate reversal costs differently.

## Open trades

If the test ends while a position remains active, the trade receives:

```text
status = "Open"
```

Only closed trades are used for completed-trade statistics.

That is why buy and hold can report zero completed trades: it entered once and never exited before the dataset ended.

# Part IV — The strategies

---

# 17. Buy and hold

Exact code:

```python
def buy_and_hold_strategy(prices):
    prices = _clean_prices(prices)
    return pd.DataFrame({"signal": 1.0}, index=prices.index)
```

The target position is always 1.

After the execution delay, the strategy enters and remains fully invested.

Buy and hold is the project benchmark because it is:

- simple
- low turnover
- easy to reproduce
- a realistic alternative to market timing

## Benchmark logic

A strategy that earns a positive return may still be unattractive.

If:

```text
Strategy return = 80%
Buy-and-hold return = 250%
```

the strategy made money but failed to beat the simple benchmark.

The comparison should include risk as well as return. A lower-return strategy may still be useful if it meaningfully reduces drawdown or volatility.

# 18. Moving-average trend following

Exact indicator code:

```python
short_average = prices.rolling(
    short_window,
    min_periods=short_window,
).mean()

long_average = prices.rolling(
    long_window,
    min_periods=long_window,
).mean()

signal = (short_average > long_average).astype(float)
```

The strategy holds the asset when the faster average is above the slower average.

## Moving-average formula

For window length $n$:

$$
MA_t(n)=\frac{1}{n}\sum_{i=0}^{n-1}P_{t-i}
$$

A shorter window gives more weight to recent information indirectly because old observations leave the window sooner.

A longer window changes more slowly.

In [10]:
ma_prices = pd.Series(
    [100, 101, 102, 101, 103, 105, 107, 106, 108, 111],
    index=pd.date_range("2025-01-01", periods=10),
)

short_ma = ma_prices.rolling(3, min_periods=3).mean()
long_ma = ma_prices.rolling(5, min_periods=5).mean()
ma_signal = (short_ma > long_ma).astype(float)

pd.DataFrame({
    "price": ma_prices,
    "short_average": short_ma,
    "long_average": long_ma,
    "target_signal": ma_signal,
})

,price,short_average,long_average,target_signal
2025-01-01,100,NaN,NaN,0.000000
2025-01-02,101,NaN,NaN,0.000000
2025-01-03,102,101.000000,NaN,0.000000
2025-01-04,101,101.333333,NaN,0.000000
2025-01-05,103,102.000000,101.400000,1.000000
2025-01-06,105,103.000000,102.400000,1.000000
2025-01-07,107,105.000000,103.600000,1.000000
2025-01-08,106,106.000000,104.400000,1.000000
2025-01-09,108,107.000000,105.800000,1.000000
2025-01-10,111,108.333333,107.400000,1.000000


## Economic idea

The strategy is a form of **trend following**:

> If the recent trend is stronger than the longer trend, remain exposed.

It does not predict a fair value. It reacts to direction.

## Why it may work

- Trends may persist.
- Investors may react gradually.
- Large institutions may adjust positions slowly.
- Market behaviour can create serial continuation.

## Why it may fail

- Averages lag prices.
- Sudden crashes happen before a crossover.
- Sudden recoveries may occur while the strategy is in cash.
- Sideways markets create whipsaws.
- Parameters may be overfit.

## `min_periods`

The strategy uses:

```python
min_periods=window
```

A 200-day average is not produced until 200 valid observations exist.

Before that, the average is missing and the comparison evaluates to false, giving a cash signal.

# 19. Momentum

Exact code:

```python
momentum_return = prices.div(
    prices.shift(lookback_window)
).sub(1.0)

signal = (momentum_return > 0).astype(float)
```

Formula:

$$
M_t(L)=\frac{P_t}{P_{t-L}}-1
$$

The strategy holds the asset when its trailing return over $L$ periods is positive.

In [11]:
momentum_prices = pd.Series(
    [100, 101, 99, 104, 106, 103, 110],
    index=pd.date_range("2025-01-01", periods=7),
)

lookback = 3
trailing_return = momentum_prices / momentum_prices.shift(lookback) - 1
momentum_signal = (trailing_return > 0).astype(float)

pd.DataFrame({
    "price": momentum_prices,
    "trailing_return": trailing_return,
    "target_signal": momentum_signal,
})

,price,trailing_return,target_signal
2025-01-01,100,NaN,0.000000
2025-01-02,101,NaN,0.000000
2025-01-03,99,NaN,0.000000
2025-01-04,104,0.040000,1.000000
2025-01-05,106,0.049505,1.000000
2025-01-06,103,0.040404,1.000000
2025-01-07,110,0.057692,1.000000


## Momentum versus moving average

Both are trend strategies, but their definitions differ.

### Moving average

Compares two smoothed price levels.

### Momentum

Compares today's price directly with one earlier price.

They can disagree because one uses averages and the other uses an endpoint return.

## Typical payoff shape

Momentum may produce:

- many small losing trades in choppy periods
- fewer large gains during sustained trends

Therefore, low trade win rate does not automatically mean negative profitability.

# 20. Mean reversion and z-scores

Mean reversion assumes that an unusually low price relative to its recent local distribution may move back toward its rolling average.

The project is **long-only**:

- Enter when price is unusually low.
- Exit when the z-score recovers.
- It does not short unusually high prices.

## Rolling mean

```python
rolling_mean = prices.rolling(
    lookback_window,
    min_periods=lookback_window,
).mean()
```

$$
\mu_t=\frac{1}{L}\sum_{i=0}^{L-1}P_{t-i}
$$

## Rolling standard deviation

```python
rolling_std = prices.rolling(
    lookback_window,
    min_periods=lookback_window,
).std(ddof=0)
```

The standard deviation measures dispersion around the rolling mean.

`ddof=0` treats the observations in the rolling window as the full population for that window.

## Z-score

```python
z_score = prices.sub(rolling_mean).div(
    rolling_std.replace(0.0, np.nan)
)
```

$$
z_t=\frac{P_t-\mu_t}{\sigma_t}
$$

Interpretation:

- $z=0$: price equals the rolling mean
- $z=-2$: price is two rolling standard deviations below the mean
- $z=+2$: price is two rolling standard deviations above the mean

In [12]:
mr_prices = pd.Series(
    [100, 101, 100, 99, 100, 90, 92, 96, 100, 102],
    index=pd.date_range("2025-01-01", periods=10),
)

lookback = 5
rolling_mean = mr_prices.rolling(lookback, min_periods=lookback).mean()
rolling_std = mr_prices.rolling(lookback, min_periods=lookback).std(ddof=0)
z_score = (mr_prices - rolling_mean) / rolling_std.replace(0.0, np.nan)

pd.DataFrame({
    "price": mr_prices,
    "rolling_mean": rolling_mean,
    "rolling_std": rolling_std,
    "z_score": z_score,
})

,price,rolling_mean,rolling_std,z_score
2025-01-01,100,NaN,NaN,NaN
2025-01-02,101,NaN,NaN,NaN
2025-01-03,100,NaN,NaN,NaN
2025-01-04,99,NaN,NaN,NaN
2025-01-05,100,100.000000,0.632456,0.000000
2025-01-06,90,98.000000,4.049691,-1.975459
2025-01-07,92,96.200000,4.308132,-0.974901
2025-01-08,96,95.400000,3.878144,0.154713
2025-01-09,100,95.600000,4.079216,1.078639
2025-01-10,102,96.000000,4.560702,1.315587


## Entry and exit rules

Exact state logic:

```python
if pd.isna(value):
    current_position = 0.0
elif current_position == 0.0 and value <= -entry_z_score:
    current_position = 1.0
elif current_position == 1.0 and value >= exit_z_score:
    current_position = 0.0
```

Example parameters:

```text
entry_z_score = 2.0
exit_z_score = 0.0
```

Then:

- Enter long at $z \le -2$
- Stay long while the recovery is incomplete
- Exit at $z \ge 0$

## State and hysteresis

The signal depends on both:

- the current z-score
- whether the strategy is already invested

This is a **stateful** rule.

Using separate entry and exit thresholds creates **hysteresis**. It prevents the strategy from entering and exiting around one identical boundary every day.

The code remembers `current_position`.

## Important statistical warning

A z-score does not prove a price is “cheap,” and it does not guarantee a normal distribution.

Price levels can trend and may not be stationary.

Mean reversion is often more statistically defensible for:

- spreads
- residuals
- relative-value relationships
- stationary processes

Using a rolling price z-score on a long-term trending ETF is an educational strategy, not proof of a stable equilibrium.

# Part V — Performance and risk statistics

---

# 21. Final value and total return

Final value:

```python
final_value = float(portfolio.iloc[-1])
```

Total return:

```python
total_return = final_value / initial_capital - 1.0
```

$$
R_{total}=\frac{V_T}{V_0}-1
$$

If £10,000 becomes £18,000:

$$
\frac{18{,}000}{10{,}000}-1=80\%
$$

# 22. Annualized return

The engine first estimates elapsed years.

For a `DatetimeIndex`:

```python
elapsed_years = (
    end_date - start_date
).total_seconds() / seconds_per_year
```

Then:

```python
annualized_return = (
    final_value / initial_capital
) ** (1.0 / elapsed_years) - 1.0
```

$$
R_{annual}=
\left(\frac{V_T}{V_0}\right)^{1/T}-1
$$

This is the compound annual growth rate.

In [13]:
initial_value = 10_000
final_value = 20_000
years = 8

cagr = (final_value / initial_value) ** (1 / years) - 1
cagr

0.09050773266525769

## Why elapsed calendar time is used

The project uses actual date distance when dates are available, rather than merely dividing observation count by 252.

This gives a more accurate duration when the dataset has missing dates or spans partial years.

For non-date indices, it falls back to:

$$
T=\frac{\text{number of periods}}{\text{periods per year}}
$$

# 23. Volatility

The project calculates sample standard deviation:

```python
daily_volatility = strategy_returns.std(ddof=1)
```

Then annualizes it:

```python
annualized_volatility = (
    daily_volatility * np.sqrt(periods_per_year)
)
```

$$
\sigma_{annual}=\sigma_{daily}\sqrt{252}
$$

## Why square-root-of-time?

If returns are independent with stable variance:

$$
Var\left(\sum_{t=1}^{n}r_t\right)=n\sigma^2
$$

Taking the square root gives:

$$
\sigma_n=\sigma\sqrt{n}
$$

This is an approximation. Financial returns can have autocorrelation and volatility clustering, so real scaling may differ.

## What volatility measures

Volatility measures dispersion, not only losses.

A large positive day increases volatility just as a large negative day does.

That is why the project also calculates drawdown and Sortino ratio.

# 24. Risk-free rate and excess return

The config contains:

```python
annual_risk_free_rate
```

The engine converts it to a daily compound-equivalent rate:

```python
daily_risk_free_rate = (
    1.0 + annual_risk_free_rate
) ** (1.0 / periods_per_year) - 1.0
```

$$
r_{f,daily}=(1+r_{f,annual})^{1/252}-1
$$

Then:

```python
excess_returns = strategy_returns - daily_risk_free_rate
```

Excess return is return above the risk-free alternative.

## Subtle project assumption

The risk-free rate is used in Sharpe and Sortino calculations.

It is **not** credited to portfolio cash.

Therefore, when the strategy position is zero, portfolio return is zero before transaction costs, even if `annual_risk_free_rate` is positive.

A more realistic engine could let idle cash earn the risk-free rate.

# 25. Sharpe ratio

Project implementation:

```python
sharpe_ratio = (
    excess_returns.mean()
    / daily_volatility
    * np.sqrt(periods_per_year)
)
```

$$
Sharpe=
\frac{\overline{r_p-r_f}}
{\sigma(r_p)}
\sqrt{252}
$$

It measures average excess return per unit of total volatility.

## Interpretation

A larger Sharpe ratio means more average excess return relative to volatility.

Rough educational guide:

| Sharpe | Possible reading |
|---:|---|
| Below 0 | Negative excess performance |
| 0–1 | Weak to moderate |
| 1–2 | Strong |
| 2–3 | Very strong |
| Above 3 | Exceptional or possibly suspicious |

These are not laws. Sharpe ratios depend heavily on sample period, asset, frequency, assumptions, and data quality.

## Limitations

Sharpe ratio:

- treats upside and downside volatility equally
- assumes the mean and standard deviation summarize performance adequately
- can be distorted by non-normal returns
- can look high for illiquid or smoothed assets
- can be overfit
- does not reveal drawdown duration

# 26. Downside deviation and Sortino ratio

The project isolates negative strategy returns:

```python
negative_returns = strategy_returns[
    strategy_returns < 0
]
```

Then:

```python
downside_deviation = negative_returns.std(ddof=1)
```

Sortino:

```python
sortino_ratio = (
    excess_returns.mean()
    / downside_deviation
    * np.sqrt(periods_per_year)
)
```

The idea is to penalize harmful volatility rather than all volatility.

## Important implementation detail

There are multiple Sortino definitions.

The project uses the standard deviation of the subset of negative returns.

Another common method calculates the root mean square of all downside deviations relative to a minimum acceptable return.

Therefore, the project's Sortino ratio should be described as the engine's **simplified downside-risk implementation**, not the only possible definition.

# 27. Maximum drawdown

The engine stores the full drawdown series and reports:

```python
max_drawdown = results["drawdown"].min()
```

The most negative observation is the maximum historical peak-to-trough decline.

A less negative value is preferable, all else equal.

But maximum drawdown does not show:

- how long the drawdown lasted
- how often large drawdowns occurred
- whether recovery happened before the dataset ended

# 28. Daily win rate

Exact code:

```python
active_returns = strategy_returns[
    positions != 0
]

daily_win_rate = (
    active_returns > 0
).mean()
```

This asks:

> On days when the executed position was non-zero, what fraction of net strategy returns were positive?

## Subtle consequence

An exit day has position `0`, but it may contain an exit transaction cost.

Because daily win rate filters using `positions != 0`, that exit-cost day is excluded.

This is the exact behaviour of the project and is a reasonable reminder that metric definitions must be documented.

# 29. Market exposure

Exact code:

```python
market_exposure = positions.abs().mean()
```

$$
Exposure=\frac{1}{N}\sum_{t=1}^{N}|w_t|
$$

With positions limited to -1, 0, and 1:

- 100% means always long or short
- 50% means exposed for half the observations
- 0% means always cash

Absolute value means long and short both count as market exposure.

## Why exposure matters

A strategy with a 10% return while exposed only 30% of the time is different from a strategy with a 10% return while exposed 100% of the time.

Exposure helps explain:

- how much risk was active
- why volatility may be lower
- how much time was spent in cash

# 30. Orders, trades, and trade win rate

## Number of orders

```python
number_of_orders = round(turnover.sum())
```

Because a reversal has turnover 2, it counts as two units of trading activity.

## Number of completed trades

Only rows in the extracted trade table with:

```text
status == "Closed"
```

are counted.

## Trade win rate

```python
(closed_trades["trade_return"] > 0).mean()
```

$$
WinRate=
\frac{\text{profitable closed trades}}
{\text{all closed trades}}
$$

## Why orders and trades differ

One long trade normally has:

- one entry order
- one exit order

So one completed trade may create two orders.

A direct reversal can close one trade and open another on the same date, producing turnover 2.

# 31. Average, best, and worst trade

The engine calculates:

```python
closed_trades["trade_return"].mean()
closed_trades["trade_return"].max()
closed_trades["trade_return"].min()
```

These reveal payoff shape.

A low win rate may be acceptable if:

- average winning trade is much larger than average loss
- best trades are large
- worst losses are controlled

# 32. Profit factor

The engine separates winning and losing closed trades.

```python
gross_profit = winning_trade_returns.sum()
gross_loss = -losing_trade_returns.sum()
profit_factor = gross_profit / gross_loss
```

$$
ProfitFactor=
\frac{\sum \text{positive trade returns}}
{|\sum \text{negative trade returns}|}
$$

Interpretation:

- Greater than 1: total winning return magnitude exceeds total losing magnitude
- Equal to 1: equal
- Below 1: losses dominate

## Project-specific caveat

The project sums percentage trade returns, not cash profits.

If trade sizes or starting capital varied, a cash-P&L profit factor could differ.

Because this engine uses full-notional positions and sequential compounding, the percentage-based value is useful but should be labelled clearly.

## Complete miniature backtest demonstration

The following cell reproduces the central accounting logic in a compact form.

In [14]:
dates = pd.date_range("2025-01-01", periods=8)

prices = pd.Series(
    [100, 102, 105, 103, 99, 101, 104, 102],
    index=dates,
    name="price",
)

target_position = pd.Series(
    [0, 1, 1, 0, -1, -1, 0, 1],
    index=dates,
    name="target_position",
    dtype=float,
)

execution_delay = 1
cost_rate = 0.001
initial_capital = 10_000.0

asset_return = prices.pct_change(fill_method=None).fillna(0.0)
position = target_position.shift(execution_delay).fillna(0.0)
previous_position = position.shift(1).fillna(0.0)
turnover = (position - previous_position).abs()
gross_strategy_return = position * asset_return
transaction_cost = turnover * cost_rate
strategy_return = gross_strategy_return - transaction_cost
portfolio = initial_capital * (1 + strategy_return).cumprod()
running_peak = portfolio.cummax()
drawdown = portfolio / running_peak - 1

mini_results = pd.DataFrame({
    "price": prices,
    "target_position": target_position,
    "position": position,
    "asset_return": asset_return,
    "gross_strategy_return": gross_strategy_return,
    "turnover": turnover,
    "transaction_cost": transaction_cost,
    "strategy_return": strategy_return,
    "portfolio": portfolio,
    "drawdown": drawdown,
})

mini_results.round(5)

,price,target_position,position,asset_return,gross_strategy_return,turnover,transaction_cost,strategy_return,portfolio,drawdown
2025-01-01,100,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,"10,000.000000",0.000000
2025-01-02,102,1.000000,0.000000,0.020000,0.000000,0.000000,0.000000,0.000000,"10,000.000000",0.000000
2025-01-03,105,1.000000,1.000000,0.029410,0.029410,1.000000,0.001000,0.028410,"10,284.117650",0.000000
2025-01-04,103,0.000000,1.000000,-0.019050,-0.019050,0.000000,0.000000,-0.019050,"10,088.229690",-0.019050
2025-01-05,99,-1.000000,0.000000,-0.038830,-0.000000,1.000000,0.001000,-0.001000,"10,078.141460",-0.020030
2025-01-06,101,-1.000000,-1.000000,0.020200,-0.020200,1.000000,0.001000,-0.021200,"9,864.464500",-0.040810
2025-01-07,104,0.000000,-1.000000,0.029700,-0.029700,0.000000,0.000000,-0.029700,"9,571.460610",-0.069300
2025-01-08,102,1.000000,0.000000,-0.019230,-0.000000,1.000000,0.001000,-0.001000,"9,561.889150",-0.070230


Read one row at a time:

1. `asset_return` measures the asset's price change.
2. `target_position` is the strategy's instruction.
3. `position` is the delayed instruction actually held.
4. `gross_strategy_return` applies exposure to the asset return.
5. `turnover` measures the change in exposure.
6. `transaction_cost` penalizes trading.
7. `strategy_return` is the net result.
8. `portfolio` compounds the net result.
9. `drawdown` compares wealth with its previous peak.

# Part VI — Research design

---

# 33. Why a benchmark is necessary

The project runs all strategies beside buy and hold.

Without a benchmark, the statement:

```text
The strategy made 80%.
```

has little context.

Questions that require a benchmark:

- Did the strategy outperform the asset?
- Did it reduce risk?
- Did it earn a better Sharpe ratio?
- Was the added complexity worthwhile?
- Did trading costs destroy the advantage?

# 34. Training, testing, and out-of-sample evidence

The project splits prices:

```python
training_prices = prices[
    prices.index < split_date
]

testing_prices = prices[
    prices.index >= split_date
]
```

## Training period

Used to search parameters.

Also called **in-sample** data.

## Testing period

Used after parameters have been selected.

Also called **out-of-sample** data.

## The scientific idea

Training data is where the model is developed.

Testing data acts as unseen evidence.

If a strategy performs well in training but badly in testing, the training pattern may have been noise or regime-specific.

# 35. Warm-up history and the test boundary

The testing suite is called with:

```python
signal_history_prices=prices
```

This means indicators for testing dates are calculated using the full earlier price history.

That is valid because prices before a testing date were already known.

Without warm-up history:

- a 200-day average would be missing for roughly 200 testing observations
- a 252-day momentum signal would be missing for roughly one trading year

## Exact boundary behaviour in the finished project

The indicator history is preserved, but `run_backtest` re-shifts the signals **inside the testing slice**:

```python
positions = target_positions.shift(
    execution_delay
).fillna(0.0)
```

Therefore, the first testing observation starts with position 0.

The project does **not** carry an already-open training-period position into the first testing return.

This creates a clean standalone test beginning from fresh capital and cash, but it is slightly different from simulating one continuous live portfolio across the split.

A professional system should explicitly choose between:

1. **Fresh-start out-of-sample test**
2. **Continuous portfolio with carried position and state**

# 36. Parameter grids

The project tests predefined candidate values.

Examples:

```python
MOVING_AVERAGE_SHORT_WINDOWS = [20, 50, 100]
MOVING_AVERAGE_LONG_WINDOWS = [100, 150, 200, 250]

MOMENTUM_LOOKBACK_WINDOWS = [63, 126, 189, 252]

MEAN_REVERSION_LOOKBACK_WINDOWS = [10, 20, 40, 60]
MEAN_REVERSION_ENTRY_Z_SCORES = [1.0, 1.5, 2.0]
MEAN_REVERSION_EXIT_Z_SCORES = [0.0, 0.5]
```

A **grid search** evaluates every valid combination.

## Constraint filtering

Moving average combinations require:

```python
short < long
```

Mean-reversion combinations require:

```python
exit_z_score < entry_z_score
```

These constraints remove logically invalid parameter sets.

# 37. Selection by Sharpe ratio

The search results are sorted by:

```python
["sharpe_ratio", "annualized_return"]
```

descending.

Therefore:

1. Highest training Sharpe wins.
2. Annualized return breaks a tie.

This encodes a research preference:

> Prioritize risk-adjusted performance, then raw compound growth.

## Why the objective function matters

Selecting by total return may favour high-risk strategies.

Selecting by maximum drawdown may favour strategies that stay in cash.

Selecting by Sharpe favours average return relative to volatility.

There is no neutral optimization target. The target expresses what “best” means.

# 38. Minimum trade filters

The project asks for minimum completed trades:

| Strategy | Preferred minimum |
|---|---:|
| Moving average | 1 |
| Momentum | 2 |
| Mean reversion | 3 |

Why?

A Sharpe ratio based on very few decisions may be fragile.

The search first filters for enough completed trades.

If no parameter combination meets the minimum, the code falls back to all results instead of failing immediately.

## Statistical caution

Even three trades are far too few for strong inference.

The minimums are practical safeguards for an educational project, not proof of statistical significance.

# 39. Overfitting and multiple testing

Overfitting means adapting a strategy too closely to historical noise.

Grid search creates **multiple-testing risk**:

- Try enough combinations.
- Some will look good by chance.
- Selecting the winner creates an upward-biased estimate.

This is also called:

- data snooping
- backtest overfitting
- selection bias

## Common overfitting signs

- Extremely high training Sharpe
- Large drop in testing performance
- One exact parameter combination dominates while nearby values fail
- Very few trades
- Complex rules without economic justification
- Repeatedly checking the testing period and changing the strategy

## Better controls

- Use a genuinely untouched test set
- Use walk-forward validation
- Test multiple assets and regimes
- Prefer simple rules
- Examine parameter stability
- Correct for multiple comparisons
- Include realistic costs
- Report all tried models, not only the winner
- Avoid tuning after viewing testing results

# 40. Full-period results versus independent evidence

The project calculates:

- Full-period results
- Training results
- Testing results

The full period includes dates used for parameter selection.

Therefore, full-period performance is **descriptive**, not independent validation.

The testing period is the main evidence because its returns were not used to choose the parameters.

<div style="padding:14px 18px;border-left:5px solid #8250df;background:#fbefff;">
<b>Correct research language</b><br>
“The selected strategy performed X out of sample under these assumptions” is stronger and more honest than “the strategy works.”
</div>

# Part VII — Engineering and limitations

---

# 41. Exports and charts

The project exports:

- Parameter-search CSVs
- Selected-parameter JSON
- Summary CSVs
- Daily-result CSVs
- Trade CSVs
- Portfolio charts
- Drawdown charts
- Moving-average indicator chart

Why export results?

- Auditability
- Reproducibility
- Easier debugging
- Comparison in spreadsheets
- Evidence for a portfolio project

## Portfolio chart

Plots wealth over time.

Useful for:

- growth
- stagnation
- compounding
- strategy divergence

## Drawdown chart

Plots percentage distance below the previous peak.

Useful for:

- crisis behaviour
- recovery
- risk comparison
- time underwater

## Indicator chart

Plots price and selected moving averages, plus the train/test split.

Useful for interpreting what the strategy saw.

# 42. Automated tests

The test suite uses synthetic prices rather than live Yahoo data.

This is important because unit tests should be:

- deterministic
- fast
- reproducible
- independent of network failures

## What each project test proves

### Signal delay

Confirms a signal does not earn the same period's return.

### Entry and exit costs

Confirms turnover and two-sided transaction costs are applied.

### Drawdown

Confirms drawdown uses the previous running peak.

### Long and short

Confirms both sides are supported.

### Invalid moving-average windows

Confirms logically invalid parameters are rejected.

### Momentum signal

Confirms positive trailing return creates a long signal.

### Mean-reversion signal

Confirms the state machine only returns valid positions.

### Invalid positions

Confirms fractional targets such as 0.5 are rejected.

# 43. Validation and defensive programming

The engine validates:

- positive initial capital
- cost rate between 0 and 1
- risk-free rate above -100%
- positive periods per year
- non-negative execution delay
- price input is a pandas Series
- at least two valid prices
- all prices are positive
- positions are only -1, 0, or 1
- daily net return is above -100%

These checks are part of financial correctness.

Bad inputs can produce plausible-looking but meaningless output if they are not rejected.

# 44. Every important modelling limitation

## 1. Daily resolution

Intraday paths are ignored.

## 2. Closing-price execution abstraction

Exact fills at or around the close are not modelled.

## 3. Constant transaction-cost rate

Real spread and slippage vary.

## 4. No volume or liquidity constraint

The strategy can trade full notional regardless of market depth.

## 5. No market impact

The order does not move the price.

## 6. Full-notional discrete positions

Only -1, 0, and 1 are allowed.

## 7. No portfolio of multiple assets

The engine accepts one price series.

## 8. Cash earns zero

Risk-free rate is used only for ratios.

## 9. Simplified short selling

No borrow fees, recalls, margin, or availability.

## 10. No taxes

Turnover may have tax consequences.

## 11. No leverage or margin model

Position magnitude is fixed at one.

## 12. No dividends as separate cash flows

The model relies on adjusted prices.

## 13. Provider risk

Historical data may be revised, missing, or incorrectly adjusted.

## 14. Survivorship and selection bias

Testing only assets known to be successful today can bias conclusions.

## 15. Parameter search bias

The best training model is selected from multiple attempts.

## 16. Regime instability

Market behaviour changes over time.

## 17. No statistical confidence intervals

A point estimate does not show uncertainty.

## 18. Test-boundary position reset

Indicators use warm-up history, but the test starts from cash.

## 19. Simplified Sortino definition

The engine uses standard deviation of negative observations only.

## 20. Percentage-based profit factor

Trade returns are summed rather than cash P&L.

# 45. How a professional engine would improve this

Possible improvements:

1. Bid, ask, spread, and slippage models
2. Open/close execution choices
3. Position sizing from 0% to 100% or leverage
4. Multiple assets and portfolio weights
5. Cash interest
6. Borrow fees and short constraints
7. Corporate-action validation
8. Volume and liquidity limits
9. Walk-forward optimization
10. Bootstrap confidence intervals
11. Monte Carlo analysis
12. Factor and beta decomposition
13. Benchmark-relative metrics
14. Drawdown duration and recovery statistics
15. Calmar ratio
16. Value at Risk and Expected Shortfall
17. Trade expectancy and payoff ratio
18. Exposure-adjusted returns
19. Continuous state across train/test boundaries
20. Event-driven execution rather than vectorized daily bars

## Optional: run the actual project engine on synthetic data

This cell works when the notebook is placed in the same folder as `backtester.py` and `strategies.py`.

In [15]:
try:
    from backtester import BacktestConfig, run_backtest
    from strategies import (
        buy_and_hold_strategy,
        mean_reversion_strategy,
        momentum_strategy,
        moving_average_strategy,
    )

    project_imports_available = True
    print("Project modules imported successfully.")
except ImportError as exc:
    project_imports_available = False
    print("Project modules were not found in the notebook's current folder.")
    print("Place this notebook inside the finished project folder to run this section.")
    print(exc)

Project modules imported successfully.


In [16]:
if project_imports_available:
    synthetic_prices = pd.Series(
        [100, 101, 102, 104, 103, 100, 96, 94, 97, 101, 105, 108],
        index=pd.date_range("2025-01-01", periods=12),
        name="Synthetic Asset",
        dtype=float,
    )

    strategy_frame = moving_average_strategy(
        synthetic_prices,
        short_window=3,
        long_window=5,
    )

    config = BacktestConfig(
        initial_capital=10_000,
        transaction_cost_rate=0.001,
        annual_risk_free_rate=0.0,
        periods_per_year=252,
        execution_delay=1,
    )

    actual_results, actual_metrics, actual_trades = run_backtest(
        synthetic_prices,
        strategy_frame["signal"],
        config,
    )

    print("Metrics")
    display(pd.Series(actual_metrics).to_frame("value"))

    print("\nDaily results")
    display(actual_results.round(5))

    print("\nTrades")
    display(actual_trades)
else:
    print("Skipped because project modules are unavailable.")

Metrics


,value
final_value,"9,557.932585"
total_return,-0.044207
annualized_return,-0.777159
annualized_volatility,0.266951
sharpe_ratio,-3.426282
sortino_ratio,-2.841588
max_drawdown,-0.069852
daily_win_rate,0.333333
market_exposure,0.250000
number_of_orders,3.000000



Daily results


,price,target_position,position,asset_return,gross_strategy_return,turnover,transaction_cost,strategy_return,portfolio,drawdown
2025-01-01,100.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,"10,000.000000",0.000000
2025-01-02,101.000000,0.000000,0.000000,0.010000,0.000000,0.000000,0.000000,0.000000,"10,000.000000",0.000000
2025-01-03,102.000000,0.000000,0.000000,0.009900,0.000000,0.000000,0.000000,0.000000,"10,000.000000",0.000000
2025-01-04,104.000000,0.000000,0.000000,0.019610,0.000000,0.000000,0.000000,0.000000,"10,000.000000",0.000000
2025-01-05,103.000000,1.000000,0.000000,-0.009620,-0.000000,0.000000,0.000000,-0.000000,"10,000.000000",0.000000
2025-01-06,100.000000,1.000000,1.000000,-0.029130,-0.029130,1.000000,0.001000,-0.030130,"9,698.737860",-0.030130
2025-01-07,96.000000,0.000000,1.000000,-0.040000,-0.040000,0.000000,0.000000,-0.040000,"9,310.788350",-0.068920
2025-01-08,94.000000,0.000000,0.000000,-0.020830,-0.000000,1.000000,0.001000,-0.001000,"9,301.477560",-0.069850
2025-01-09,97.000000,0.000000,0.000000,0.031910,0.000000,0.000000,0.000000,0.000000,"9,301.477560",-0.069850
2025-01-10,101.000000,0.000000,0.000000,0.041240,0.000000,0.000000,0.000000,0.000000,"9,301.477560",-0.069850



Trades


,entry_date,exit_date,side,entry_price,exit_price,holding_days,trade_return,status
0,2025-01-06,2025-01-08,Long,100.000000,94.000000,2,-0.069852,Closed
1,2025-01-12,2025-01-12,Long,108.000000,108.000000,1,0.027571,Open


# Revision

---

# 46. Code-to-theory map

| Code | Theory |
|---|---|
| `prices.pct_change()` | Simple close-to-close return |
| `target_positions.shift(1)` | Execution delay and look-ahead prevention |
| `positions * asset_returns` | Exposure-weighted gross return |
| `abs(position - previous_position)` | Turnover |
| `turnover * cost_rate` | Proportional transaction costs |
| `gross_return - costs` | Net strategy return |
| `(1 + returns).cumprod()` | Geometric compounding |
| `portfolio.cummax()` | Running wealth peak |
| `portfolio / peak - 1` | Drawdown |
| `rolling(...).mean()` | Moving average |
| `price / shifted_price - 1` | Momentum |
| `(price - mean) / std` | Z-score |
| `returns.std() * sqrt(252)` | Annualized volatility |
| `mean(excess) / std * sqrt(252)` | Sharpe ratio |
| `mean(excess) / downside_std * sqrt(252)` | Project Sortino ratio |
| `mean(abs(position))` | Market exposure |
| `sum(wins) / abs(sum(losses))` | Project profit factor |
| `prices.index < split_date` | Training data |
| `prices.index >= split_date` | Testing data |
| Grid loops | Parameter search |
| Sort by training Sharpe | Model-selection objective |

# 47. Formula sheet

## Asset return

$$
r_t=\frac{P_t}{P_{t-1}}-1
$$

## Gross strategy return

$$
r^{gross}_{s,t}=w_t r_t
$$

## Turnover

$$
TO_t=|w_t-w_{t-1}|
$$

## Transaction cost

$$
C_t=TO_t c
$$

## Net strategy return

$$
r^{net}_{s,t}=r^{gross}_{s,t}-C_t
$$

## Portfolio value

$$
V_t=V_0\prod_{i=1}^{t}(1+r^{net}_{s,i})
$$

## Total return

$$
R_{total}=\frac{V_T}{V_0}-1
$$

## Annualized return

$$
R_{annual}=
\left(\frac{V_T}{V_0}\right)^{1/T}-1
$$

## Annualized volatility

$$
\sigma_{annual}=\sigma_{period}\sqrt{N}
$$

## Periodic risk-free rate

$$
r_{f,period}=(1+r_{f,annual})^{1/N}-1
$$

## Sharpe ratio

$$
Sharpe=
\frac{\overline{r_p-r_f}}{\sigma_p}\sqrt{N}
$$

## Project Sortino ratio

$$
Sortino=
\frac{\overline{r_p-r_f}}{\sigma(r_p\mid r_p<0)}\sqrt{N}
$$

## Drawdown

$$
DD_t=\frac{V_t}{\max(V_1,\ldots,V_t)}-1
$$

## Moving average

$$
MA_t(L)=\frac{1}{L}\sum_{i=0}^{L-1}P_{t-i}
$$

## Momentum

$$
M_t(L)=\frac{P_t}{P_{t-L}}-1
$$

## Z-score

$$
z_t=\frac{P_t-\mu_t}{\sigma_t}
$$

## Market exposure

$$
Exposure=\frac{1}{N}\sum_{t=1}^{N}|w_t|
$$

## Profit factor

$$
PF=
\frac{\sum \text{winning trade returns}}
{|\sum \text{losing trade returns}|}
$$

# 48. Glossary

| Term | Meaning |
|---|---|
| Adjusted close | Historical price adjusted for corporate actions according to the data provider |
| Annualized | Converted to an equivalent yearly rate |
| Backtest | Historical simulation of a strategy |
| Benchmark | Reference strategy used for comparison |
| CAGR | Compound annual growth rate |
| Cash position | Zero exposure to the asset |
| Close-to-close | Return from one daily close to the next |
| Drawdown | Decline below a previous portfolio peak |
| Execution delay | Gap between signal creation and position application |
| Exposure | Fraction of time or capital subject to market movement |
| Gross return | Return before costs |
| Hysteresis | Different entry and exit boundaries that reduce rapid switching |
| In sample | Data used for model development |
| Long | Position that benefits from rising price |
| Look-ahead bias | Use of information not yet available |
| Mean reversion | Hypothesis that deviations may move back toward a local mean |
| Momentum | Tendency or rule based on continuation of prior movement |
| Net return | Return after modelled costs |
| Out of sample | Data not used to choose model parameters |
| Parameter | Strategy setting such as lookback length |
| Profit factor | Winning-return magnitude divided by losing-return magnitude |
| Risk-free rate | Return assumed available without market risk in the metric model |
| Sharpe ratio | Excess return per unit of total volatility |
| Short | Position that benefits from falling price |
| Signal | Strategy's desired target position |
| Slippage | Difference between expected and actual execution price |
| Sortino ratio | Excess return relative to downside risk |
| Target position | Position requested by the strategy |
| Turnover | Magnitude of change in position |
| Volatility | Dispersion of returns |
| Warm-up history | Earlier observations used to initialize indicators |
| Whipsaw | Repeated losing entries and exits in a choppy market |
| Z-score | Number of standard deviations from a mean |

# 49. Revision questions

1. Why is adjusted price preferred to raw close for historical return work?
2. What exact return interval does this engine model?
3. Why must dates be sorted before using `pct_change`?
4. What is the difference between target position and executed position?
5. What does a position of -1 mean?
6. Why is `shift(1)` essential for a close-based strategy?
7. Why is a direct long-to-short reversal turnover 2?
8. How does transaction cost enter net return?
9. Why is wealth compounded rather than returns simply added?
10. What does a -30% drawdown mean?
11. Why can buy and hold have zero completed trades?
12. What theory does a moving-average crossover represent?
13. How does momentum differ from a moving-average crossover?
14. What does a z-score of -2 mean inside the rolling window?
15. Why is the mean-reversion strategy stateful?
16. Why does the strategy use separate entry and exit z-scores?
17. What does annualized return measure?
18. Why is volatility multiplied by square root of 252?
19. How is the annual risk-free rate converted to a daily rate?
20. What does Sharpe ratio reward and penalize?
21. How does Sortino differ conceptually from Sharpe?
22. Why is daily win rate not the same as trade win rate?
23. What does market exposure measure?
24. Why may a low-win-rate strategy still be profitable?
25. What does profit factor above 1 mean?
26. Why is training performance not independent evidence?
27. What is the purpose of the testing period?
28. What is warm-up history?
29. Does the final project carry the training position into the first testing return?
30. Why does grid search create multiple-testing risk?
31. Why are full-period results called descriptive?
32. What real execution effects are missing?
33. Why do deterministic unit tests use synthetic data?
34. What is the difference between financial validity and software correctness?
35. Which metric would you inspect alongside return before judging a strategy?

# 50. Answers

1. It reduces artificial jumps caused by corporate actions according to the provider's adjustment method.
2. Previous daily close to current daily close.
3. Time-series operations use row order as chronology.
4. Target is the strategy's request; executed position is the delayed exposure applied by the engine.
5. Fully short.
6. To prevent today's completed close from deciding the position that earned today's close-to-close return.
7. One unit closes the long and another establishes the short.
8. Net return equals gross strategy return minus turnover-based cost.
9. Each period's return applies to the latest wealth level.
10. The portfolio is 30% below a previous running peak.
11. It enters once and remains open at the end.
12. Trend following.
13. Momentum compares endpoints; moving average compares smoothed price levels.
14. Price is two rolling standard deviations below the rolling mean.
15. The rule depends on whether it is already invested.
16. Hysteresis reduces repeated switching around one boundary.
17. Equivalent compound growth per year.
18. Under independent stable variance, variance scales with time and standard deviation with square root of time.
19. $(1+r_{annual})^{1/252}-1$.
20. It rewards average excess return and penalizes total volatility.
21. Sortino focuses on downside observations rather than all variation.
22. One counts profitable active days; the other counts profitable completed positions.
23. Average absolute position magnitude.
24. A few large winners can outweigh many small losses.
25. Winning trade-return magnitude exceeds losing trade-return magnitude.
26. Parameters were chosen using that data.
27. To evaluate frozen parameters on unseen history.
28. Earlier data used to initialize indicators at the test start.
29. No. Indicators are warmed up, but the sliced test starts from cash after the internal shift.
30. Some parameter combinations will look good by chance.
31. They include training dates and therefore are not independent validation.
32. Variable spread, slippage, market impact, liquidity, borrow costs, taxes, exact fills, and more.
33. To make tests fast, reproducible, and independent of networks.
34. Software can execute correctly while the financial assumptions remain unrealistic.
35. At minimum: drawdown, Sharpe/Sortino, volatility, exposure, costs, and out-of-sample performance.

<div style="padding:20px 24px;border:1px solid #d0d7de;border-radius:12px;">
<h2 style="margin-top:0;">Final mental model</h2>

A backtest is a chain of conditional accounting:

```text
Data → Signal → Delayed position → Return → Cost → Wealth → Risk metrics
```

A research workflow adds another chain:

```text
Training search → Frozen parameters → Out-of-sample test → Honest interpretation
```

The quality of the result depends on both chains. Correct Python cannot rescue an invalid financial assumption, and good finance theory cannot rescue buggy code.
</div>